# NLU LoRA fine-tune (Gemma-2B-it) — ai-voice-calling-collections-agent

Runtime → **T4 GPU**. Whole run ≈ 30–40 min. Needs a Hugging Face token with access to `google/gemma-2b-it` (accept the licence on the model page first).

In [ ]:
!nvidia-smi -L
!pip -q install -U 'transformers>=4.44' 'peft>=0.12' accelerate huggingface_hub 'torchao>=0.16'

In [ ]:
!git clone --depth 1 https://github.com/Shelunagori/ai-voice-calling-collections-agent.git
%cd ai-voice-calling-collections-agent/backend
!pip -q install -e . --no-deps
!pip -q install pydantic pydantic-settings

In [ ]:
from huggingface_hub import login

login()  # paste your HF token (read + write if you will push the adapter)

## 1. Zero-shot baseline of the base model (before)

In [ ]:
!python training/train_lora.py eval --base google/gemma-2b-it --out training/reports/before-gemma2b-zeroshot.json --md training/reports/after.md

## 1b. Smoke test (5 optimizer steps, ~2 min) — loss must be a number, not nan. If it is nan, add `--dtype fp32 --batch 2 --grad-accum 8` to the training commands.

In [ ]:
!python training/train_lora.py train --base google/gemma-2b-it --out /content/smoke --max-steps 5

## 2. Train the LoRA adapter (r=8, attention projections; ~25 min on T4)

In [ ]:
!python training/train_lora.py train --base google/gemma-2b-it --out training/adapter --epochs 3

## 3. Evaluate the adapter on the frozen held-out set (after)

In [ ]:
!python training/train_lora.py eval --base google/gemma-2b-it --adapter training/adapter --out training/reports/after-gemma2b-lora-local.json --md training/reports/after.md

## 4. Push the adapter to the Hub and download the reports

Change `REPO` to your namespace. Only `adapter_config.json`, `adapter_model.safetensors`, `training_meta.json` are uploaded.

In [ ]:
REPO = 'YOUR-HF-USER/vca-nlu-gemma-2b-lora'
!python training/train_lora.py push --adapter training/adapter --repo $REPO

In [ ]:
!zip -qr /content/adapter.zip training/adapter -x 'training/adapter/checkpoints/*'
!zip -qj /content/reports.zip training/reports/before-gemma2b-zeroshot.json training/reports/after-gemma2b-lora-local.json training/reports/after.md
from google.colab import files

files.download('/content/adapter.zip')
files.download('/content/reports.zip')